### Optimization Problem (P) - Minimum Potential Energy Formulation

$$
\begin{aligned}
(\mathscr{P}) \quad \min_{v \in U_{ad}} \quad & J(y, v) = \frac{1}{2} \|y - z_d\|_{L^2(\Omega)}^2 + \frac{V}{2} \|v\|_{L^2(\Omega)}^2 \\
\text{subject to} \quad & y = \argmin_{w \in K} \mathscr{E}(w; v) \\
& K = \{ w \in H_0^1(\Omega) \mid w \geq 0 \text{ a.e. in } \Omega \}
\end{aligned}
$$

#### Potential Energy Functional
The energy functional $\mathscr{E}(w; v)$ is given by:

$$
\mathscr{E}(w; v) = \frac{1}{2}a(w,w) - \langle f + v, w \rangle
$$

where:
- $a(w,w) = \int_\Omega \nabla w \cdot \nabla w \, dx$ (Dirichlet energy)
- $\langle f + v, w \rangle = \int_\Omega (f + v)w \, dx$ (work done by external forces)

In [ ]:
from pathlib import Path
import sys

PACKAGE_ROOT = Path.cwd().resolve()
if not (PACKAGE_ROOT / "utils.py").is_file():
    raise FileNotFoundError("Run this notebook from the final_code package directory.")
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import matplotlib.pyplot as plt
import torch
from   torch.autograd import grad
from utils import *
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader,TensorDataset
from scipy.stats.qmc import LatinHypercube
from functools import partial
from torch.optim.lr_scheduler import LambdaLR
import os
import time 
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from IPython.display import clear_output
from datetime import datetime
ROOT = PACKAGE_ROOT / "outputs" / "example1_control_constraints"

In [ ]:
class net_y_res(nn.Module):
    def __init__(self, source, hidden=64, layers=3, init=True, activation='Tanh', hard_=None, obstacle=None):
        super(net_y_res, self).__init__()
        
        # Choose activation function based on activation_flag
        self.activation = activation
        if activation == 'Tanh':
            self.activation = nn.Tanh()
        elif activation == 'Identity':
            self.activation = nn.Identity()
        elif activation == 'Square':
            self.activation = SquaredReLU()
        elif activation == 'Cubic':
            self.activation = CubicReLU()
        elif activation == 'SELU':
            self.activation = nn.SELU()
        elif activation == 'Swish':
            self.activation = nn.SiLU()
        elif activation == 'ReLU':
            self.activation = nn.ReLU()
        
        
        # Define padding size and apply zero-padding to inputs
        self.padding_size = hidden - 2  # Padding size is hidden - 2 based on your code
        
        # Residual blocks
        self.res_blocks = nn.ModuleList(
            [ResidualBlock(hidden, self.activation) for _ in range(layers - 1)]
        )
        
        # Output layer
        self.output_layer = nn.Linear(hidden, 1)
        
        self.hard = hard_
            
        self.obstacle = obstacle  
        if init:
            self._initialize_weights()

    def _initialize_weights(self):
        # Initialize weights for the output layer
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)
        
    def forward(self, x):        
        # Apply zero-padding to the input tensor
        padded_x = torch.cat([x, torch.zeros(x.size(0), self.padding_size).to(x.device)], dim=1)
        
        # Apply hard constraint (if provided)
        x1 = padded_x[:, 0:1]
        x2 = padded_x[:, 1:2]
        
        if self.hard is None:
            hard = x1 * (1 - x1) * x2 * (1 - x2)
        else:
            hard = self.hard(x1, x2)
        
        # Directly pass through residual blocks
        out = padded_x
        for res_block in self.res_blocks:
            out = res_block(out)
        
        # Output layer
        out = self.output_layer(out)
        
        # Apply hard constraint and ensure obstacle constarint
        if self.obstacle == None:
            out = out**2
        
        out = out * hard
  
        return out

# Upper- and lower-level losses: objective and energy functional

In [ ]:
def J_u(source,net_y,net_v,x):
    """
    Objective function:
    J(y, v) = 1/2 * ||y - z_d||^2 + (V/2) * ||v||^2
    Use MSELoss to approximate the integrals.
    """
    yd = source.compute_yd
    nu = source.nu
    term1 = (1/2)*torch.mean((net_y(x)-yd(x))**2)
    term2 = (nu/2) * torch.mean(net_v(x)**2)

    return term1 +term2


def J_l(source, net_y, net_v, x):
    """
    The lower-level state y minimizes the potential energy determined by control v:
    J(w, v) = 1/2*a(w,w) - <v+f,w>
    Use MSELoss to approximate the integrals.
    """
    f = source.compute_f
    x = x.clone().detach().requires_grad_(True)
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    x = torch.cat([x1, x2], 1)
    
    f_val = f(x)
    y_pred  = net_y(x)
    v_pred  = net_v(x)
    y_x1 = grad(y_pred.sum(), x1, create_graph=True)[0]
    y_x2 = grad(y_pred.sum(), x2, create_graph=True)[0]


    a_val = y_x1**2 + y_x2**2
    linear_term = (f_val+v_pred)*y_pred

    return torch.mean(0.5*a_val-linear_term)

# Relative errors

In [ ]:
def rel_l2_error_yv(net_y, net_v, source, device='cpu',resolution=101):
    """
    Compute the relative L2 errors of y and v on [0,1]×[0,1]:
       rel_l2_y = ||y_app - y_ref|| / ||y_ref||
       rel_l2_v = ||v_app - v_ref|| / ||v_ref||
    """

    x = torch.linspace(0, 1, resolution, device=device)
    X, Y = torch.meshgrid(x, x, indexing='ij')

    xy = torch.cat((X.reshape(-1,1), Y.reshape(-1,1)), dim=1)
    X, Y = torch.meshgrid(x[1:-1], x[1:-1], indexing='ij')
    xy_interior = torch.cat((X.reshape(-1,1), Y.reshape(-1,1)), dim=1)

    with torch.no_grad():
        y_star = source.compute_y_star(xy)
        v_star = source.compute_u_star(xy_interior)
    

    y_app = net_y(xy)  
    v_app = net_v(xy_interior)  


    num_y = torch.sum((y_app - y_star) ** 2)
    den_y = torch.sum(y_star ** 2)
    num_v = torch.sum((v_app - v_star) ** 2)
    den_v = torch.sum(v_star ** 2)

    # Prevent division by zero when the reference solution is zero.
    rel_l2_y = torch.sqrt(num_y / (den_y + 1e-30))
    rel_l2_v = torch.sqrt(num_v / (den_v + 1e-30))


    print(f"Relative L2 error for y: {rel_l2_y.item():.4e}")
    print(f"Relative L2 error for v: {rel_l2_v.item():.4e}")

    return rel_l2_y, rel_l2_v

In [ ]:
def get_xi(source, net_y, net_v, x):
    """
    Constraint:
    -Δy = v + f + xi
    Compute xi.
    """
    x = x.clone().detach().requires_grad_(True)
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    x = torch.cat([x1, x2], 1)
    y_pred  = net_y(x)
    v_pred  = net_v(x)


    f_val = source.compute_f(x)
    


    y_x1 = grad(y_pred.sum(), x1, create_graph=True)[0]
    y_x2 = grad(y_pred.sum(), x2, create_graph=True)[0]

    d2y_dx1 = grad(y_x1.sum(), x1, create_graph=True)[0]
    d2y_dx2 = grad(y_x2.sum(), x2, create_graph=True)[0]
    neg_lap_y = -(d2y_dx1 + d2y_dx2)

    xi = neg_lap_y - v_pred - f_val
    return xi

def ortho_loss(source,net_y, net_v, x):
    """
    Require y and xi to be orthogonal in a chosen metric.
    """
    y_pred  = net_y(x)
    xi = get_xi(source, net_y, net_v, x)

    return torch.mean((y_pred*xi)**2)

def non_neg_loss(source,net_y,net_v, x):
    """
    Require xi to be nonnegative.
    """
    xi = get_xi(source, net_y, net_v, x)
    return torch.mean((torch.relu(-xi))**2)

# Random sampling in the domain

In [ ]:
class RandomDataLoader:
    """
    Random data loader.
    """
    def __init__(self, batch_size=512, steps_per_epoch=4, dims=2, device='cpu',
                 dual_sampling=False):
        """
        Initialize the random data loader.
        
        Parameters:
            batch_size (int): Target total batch size.
            steps_per_epoch (int): Number of batches per epoch.
            dims (int): Input dimension (default: 2, for x1 and x2).
            device (str): Device for data generation (e.g. 'cpu' or 'cuda:0').
            dual_sampling (bool): Whether to generate two independent samples per iteration.
        """
        self.steps_per_epoch = steps_per_epoch
        self.dims = dims
        self.device = device
        self.current_step = 0
        

        self.interior_points = max(1, batch_size)
        

        self.batch_size = self.interior_points
        self.dual_sampling = dual_sampling
 
    def __iter__(self):
        """Return the iterator itself."""
        self.current_step = 0
        return self
    
    def __next__(self):
        """Generate a full batch on each call."""
        if self.current_step < self.steps_per_epoch:
            self.current_step += 1
            
            if self.dual_sampling:
                # Use the same points for y/z and an independent sample for u.
                yz_points = torch.rand(self.interior_points, self.dims, device=self.device)
                u_points = torch.rand(self.interior_points, self.dims, device=self.device)
                return yz_points, u_points
            
            single_points = torch.rand(self.interior_points, self.dims, device=self.device)
            return (single_points,)
        else:

            self.current_step = 0
            raise StopIteration
    
    def __len__(self):
        """Return the number of batches per epoch."""
        return self.steps_per_epoch

# Trainer

In [ ]:
class Trainer:
    def __init__(
        self,
        v: nn.Module,
        y: nn.Module,
        theta: nn.Module,
        source: Source,
        train_loader: DataLoader,
        optimizer_v: torch.optim.Optimizer,
        optimizer_y: torch.optim.Optimizer,
        optimizer_theta: torch.optim.Optimizer,
        scheduler_v: torch.optim.lr_scheduler.LambdaLR, 
        scheduler_y: torch.optim.lr_scheduler.LambdaLR,
        scheduler_theta: torch.optim.lr_scheduler.LambdaLR,
        gamma: float = 10.0,
        p: float = 0.5,
        device: torch.device = torch.device("cpu"),
        num_epochs: int = 10000,
        weight: list = [1, 1],
        save: bool = False,
        lr: float = 1e-3,
    ):    
        """
        Args:
            v (nn.Module): Control v(x).
            y (nn.Module): State y(x).
            xi (nn.Module):  Multiplier xi(x).
            source (Source): Source instance.
            train_loader (DataLoader): DataLoader.
            optimizer (torch.optim.Optimizer): Optimizer.
            scheduler (torch.optim.lr_scheduler.LambdaLR): Learning-rate scheduler.
            gamma (float): weight of Moreau envelope.
            p (float): parameter of penalty.
            0 <= p < 1.
            device (torch.device): Training device.
            num_epochs (int): Maximum number of training epochs.
            weight (list): weight of loss.
            save (bool): Whether to save the models.
        """

        self.v = v.to(device)
        self.y = y.to(device)
        self.theta = theta.to(device)
        self.source = source
        self.train_loader = train_loader
        self.optimizer_v = optimizer_v
        self.optimizer_y = optimizer_y
        self.optimizer_theta = optimizer_theta
        self.scheduler_v = scheduler_v
        self.scheduler_y = scheduler_y
        self.scheduler_theta = scheduler_theta
        self.gamma = gamma
        self.p = p
        self.device = device
        self.num_epochs = num_epochs
        self.weight = weight
        self.save = save
        self.num = 0
        self.timestart = 0
        self.record_u = torch.zeros(1).to(self.device)
        self.record_l = torch.zeros(1).to(self.device)
        self.record_ortho = torch.zeros(1).to(self.device)
        self.record_non_neg = torch.zeros(1).to(self.device)
        self.lr = lr
        self.file_suffix = f"zy_testnorestart_allusezk{datetime.now().strftime('%Y%m%d')}udecay_fixuboundary_gamma_{self.gamma}_cycle_factor_0.8_batch_size{batch_size}_{self.num_epochs}_p_{self.p}_lr_{self.lr}_yz"
        if save == True:
            os.makedirs(f'{ROOT}/{self.file_suffix}', exist_ok=True)
        if save == True:
            self.loss_csv_file = f'{ROOT}/{self.file_suffix}/loss.csv'

            with open(self.loss_csv_file, 'w') as f:
                f.write("epoch,time,J_u_loss,J_l_loss\n")
            self.norm_csv_file = f'{ROOT}/{self.file_suffix}/norm.csv'

            with open(self.norm_csv_file, 'w') as f:
                f.write("epoch,time,rel_l2_error_y,rel_l2_error_v\n")


    def diff_net(self, net_y, net_theta):
        """
        Compute the difference between two networks, considering only linear-layer weights and biases.
        """
        res = 0
        for para1, para2 in zip(net_y.parameters(), net_theta.parameters()):
            res += torch.nn.MSELoss(reduction='sum')(para1, para2)
        return res
    
    def getloss_u(self,source,net_y, net_v, x):
        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        return self.weight[0]*J_u(source, net_y, net_v, x)
    
    def getloss_l(self, source, net_y, net_v, x):

        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        

        result = self.weight[1] * J_l(source, net_y, net_v, x)
        
        
        return result
    
    def train_one_epoch(self, epoch, timestart):
        self.v.train()
        self.y.train()
        self.theta.train()

        ck = 5*(epoch)**self.p
        inv_ck = 1.0 / ck if ck > 0 else 0.0
        for batch in self.train_loader:
            if isinstance(batch, (tuple, list)):
                if len(batch) == 2:
                    x_yz_raw, x_u_raw = batch
                else:
                    x_yz_raw = batch[0]
                    x_u_raw = batch[0]
            else:
                x_yz_raw = batch
                x_u_raw = batch

            x_yz = x_yz_raw.detach().requires_grad_(True).to(self.device)

            with torch.set_grad_enabled(True):
                theta_params = list(self.theta.parameters())
                y_params = list(self.y.parameters())
                v_params = list(self.v.parameters())
                
                # 1. update theta
                loss_theta_e = self.getloss_l(self.source, self.theta, self.v, x_yz)
                grad_theta_e = torch.autograd.grad(loss_theta_e, theta_params, retain_graph=False)
                theta_total_grads = []
                for g_e, p_theta, p_y in zip(grad_theta_e, theta_params, y_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    theta_total_grads.append(g_e + prox_term)
                self._apply_custom_grads(self.theta, self.optimizer_theta, theta_total_grads)



                # 2. update y and u
                loss_upper_y = self.getloss_u(self.source, self.y, self.v, x_yz)
                loss_lower_y = self.getloss_l(self.source, self.y, self.v, x_yz)
                
                h_j_y = torch.autograd.grad(loss_upper_y, y_params, retain_graph=True)
                h_e_y = torch.autograd.grad(loss_lower_y, y_params, retain_graph=False)
                
                y_total_grads = []
                for g_j, g_e, p_y, p_theta in zip(h_j_y, h_e_y, y_params, theta_params):
                    prox_term = (p_theta.detach() - p_y.detach()) * (1.0 / self.gamma)
                    y_total_grads.append(inv_ck * g_j + g_e + prox_term)

                
                self._apply_custom_grads(self.y, self.optimizer_y, y_total_grads)
                
                x_u = x_u_raw.detach().requires_grad_(True).to(self.device)
                loss_upper_u = self.getloss_u(self.source, self.y, self.v, x_u)
                loss_lower_u = self.getloss_l(self.source, self.y, self.v, x_u)
                loss_lower_theta = self.getloss_l(self.source, self.theta, self.v, x_u)
                
                h_j_u = torch.autograd.grad(loss_upper_u, v_params, retain_graph=True)
                h_e_u = torch.autograd.grad(loss_lower_u, v_params, retain_graph=True)
                h_e_u_theta = torch.autograd.grad(loss_lower_theta, v_params, retain_graph=False)
                
                v_total_grads = []
                for g_j_u, g_e_u, g_e_u_theta in zip(h_j_u, h_e_u, h_e_u_theta):
                    v_total_grads.append(inv_ck * g_j_u + g_e_u - g_e_u_theta)

                
                self._apply_custom_grads(self.v, self.optimizer_v, v_total_grads)



        self.scheduler_v.step()
        self.scheduler_y.step()
        self.scheduler_theta.step()


        if (epoch-1) % 10 ==0:        
            # Evaluate the losses on a uniform grid.
            grid_size = 101
            x1 = torch.linspace(0, 1, grid_size, device=self.device)
            x2 = torch.linspace(0, 1, grid_size, device=self.device)
            X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
            grid_points = torch.stack([X1.flatten(), X2.flatten()], dim=1).requires_grad_(True)
            self.record_u = J_u(self.source, self.y, self.v, grid_points)
            self.record_l = J_l(self.source, self.y, self.v, grid_points)
            with open(self.loss_csv_file, 'a') as f:
                f.write(f"{epoch},{time.time()-timestart},{self.record_u.item()},{self.record_l.item()}\n")

            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        return self.record_u.item(), self.record_l.item()
    
    def _update_model(self, model, optimizer, loss):
        """Update the model using manually computed gradients."""
        optimizer.zero_grad()
        grad_params = torch.autograd.grad(loss, model.parameters())
        for param, g in zip(model.parameters(), grad_params):
            param.grad = g
        optimizer.step()
    
    def _apply_custom_grads(self, model, optimizer, grads):
        """Update model parameters using custom gradients."""
        optimizer.zero_grad()
        for param, grad in zip(model.parameters(), grads):
            param.grad = grad
        optimizer.step()
    
    def validate(self,epoch):
        self.v.eval()
        self.y.eval()
        plt.close('all')
        rel_l2_error_y, rel_l2_error_v = rel_l2_error_yv(self.y, self.v, self.source, device=self.device)
        with open(self.norm_csv_file, 'a') as f:
            f.write(f"{epoch},{time.time()-self.timestart},{rel_l2_error_y.item()},{rel_l2_error_v.item()}\n")
            
        # with torch.no_grad():
        #     if self.save == True:
        #         yfile = f'{ROOT}/{self.file_suffix}/y_net_{epoch}.png'
        #         vfile = f'{ROOT}/{self.file_suffix}/v_net_{epoch}.png'
        #         thetafile = f'{ROOT}/{self.file_suffix}/theta_net_{epoch}.png'
        #     else:
        #         yfile = None
        #         vfile = None
        #         thetafile = None
        #     visual_y_3d(self.source, self.y)
        #     visual_v_3d(self.source, self.v)
        

    def fit(self):
        pbar = tqdm(range(1, self.num_epochs + 2), ascii=True, dynamic_ncols=True)
        self.timestart = time.time()
        clear_output(wait=True)
        for epoch in pbar:

            J_u_loss, J_l_loss = self.train_one_epoch(epoch, self.timestart)


            pbar.set_postfix(
                J_u_loss=J_u_loss,
                J_l_loss=J_l_loss
            )
            if epoch == 500:
                # Raise lower clamp of v after warmup
                if self.v.constraint is None:
                    self.v.constraint = [-1.0, 0.7]
                else:
                    self.v.constraint = list(self.v.constraint)
                self.v.constraint[0] = 0.0
            if (epoch-1) % 500 == 0:
                self.validate(epoch)
            

            # if epoch % 500 == 0 and self.save == True:
            #     torch.save(self.v.state_dict(), f'{ROOT}/{self.file_suffix}/u_net_{epoch}.pth')
            #     torch.save(self.y.state_dict(), f'{ROOT}/{self.file_suffix}/y_net_{epoch}.pth')
            #     torch.save(self.theta.state_dict(), f'{ROOT}/{self.file_suffix}/theta_net_{epoch}.pth')

# Device

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
source = Source5(device=device,scale=20.0)
set_seed(123)

# Hyperparameters

In [ ]:
# hyperparameters
lr = 1e-3
warmup = 100
cycle_factor=0.8
cycle_length=1000
num_epochs=20000
weight = [1, 1]
save= True
batch_size = 512

In [ ]:
def hard(x1, x2):
    return x1 * (1 - x1) * x2 * (1 - x2)

## Bilevel MEHA training

In [ ]:
pretrain_loader = RandomDataLoader(
    batch_size=batch_size,
    steps_per_epoch=1,
    device=device
)

train_loader = RandomDataLoader(
    batch_size=batch_size,
    steps_per_epoch=4,
    device=device,
    dual_sampling=True  # Use shared samples for y/theta and independent samples for v.
)
y = net_y_res(source=source,hidden=16,layers=4,activation = 'Swish').to(device)
v = net_v_res(source=source,hidden=16,layers=4,activation = 'Swish',hard_=hard,constraint=[-1,0.7]).to(device)
theta = net_y_res(source=source,hidden=16,layers=4,activation = 'Swish').to(device)
pretrain_dir = os.path.join(ROOT, "pretrain")
os.makedirs(pretrain_dir, exist_ok=True)

y = pretrain_y_net(
    net_y=y,
    source=source,
    dataloader=pretrain_loader,
    device=device,
    num_epochs=100,
    lr=1e-4,
    warmup=0,
    cycle_factor=0.9,
    cycle_length=100,
    save_dir=os.path.join(pretrain_dir, 'y_net'))

theta.load_state_dict(y.state_dict())


optimizer_y = torch.optim.Adam(y.parameters(), lr=lr, betas=(0.99, 0.999))
optimizer_v = torch.optim.Adam(v.parameters(), lr=lr)
optimizer_theta = torch.optim.Adam(theta.parameters(), lr=lr, betas=(0.99, 0.999))
scheduler_y = LambdaLR(optimizer_y,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))
scheduler_v = LambdaLR(optimizer_v,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))
scheduler_theta = LambdaLR(optimizer_theta,partial(lr_lambda,warmup=warmup,cycle_factor=cycle_factor,cycle_length=cycle_length))

trainer = Trainer(
    v=v,
    y=y,
    theta=theta,
    source=source,
    train_loader=train_loader,
    optimizer_v=optimizer_v,
    optimizer_y=optimizer_y,
    optimizer_theta=optimizer_theta,
    scheduler_v=scheduler_v,
    scheduler_y=scheduler_y,
    scheduler_theta=scheduler_theta,
    gamma=20.0,
    p=0.3,
    device=device,
    num_epochs=num_epochs,
    weight=weight,
    save=save,
    lr = lr
)
trainer.fit()

In [ ]:
import os

# Save the final models in the models directory alongside loss.csv and norm.csv.
model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
os.makedirs(model_dir, exist_ok=True)
torch.save(y.state_dict(), os.path.join(model_dir, "final_y.pth"))
torch.save(v.state_dict(), os.path.join(model_dir, "final_v.pth"))
torch.save(theta.state_dict(), os.path.join(model_dir, "final_theta.pth"))

## After bilevel training, fix v and solve the lower-level energy minimization separately to obtain an accurate y

In [ ]:
def clone_trainer_and_train_fixed_v(original_trainer, additional_epochs=5000):
    """
    Create a clone of the trainer and continue training with v fixed, optimizing y using J_l loss.
    
    Args:
        original_trainer: The original trained Trainer instance
        additional_epochs: Number of additional epochs for the fixed-v training
        
    Returns:
        cloned_trainer: The cloned trainer instance after additional training
    """
    import copy
    import torch
    from tqdm import tqdm
    import time
    
    # Create a shallow copy of the trainer
    cloned_trainer = copy.copy(original_trainer)
    
    # Deep copy the networks (important to not affect original networks)
    cloned_trainer.y = copy.deepcopy(original_trainer.y)
    cloned_trainer.v = copy.deepcopy(original_trainer.v)
    cloned_trainer.theta = copy.deepcopy(original_trainer.theta) if original_trainer.theta is not None else None
    
    # Create fresh optimizers with the same parameters
    lr_y = original_trainer.optimizer_y.param_groups[0]['lr']
    cloned_trainer.optimizer_y = torch.optim.Adam(cloned_trainer.y.parameters(), lr=lr_y* 0.1) 
    
    # Copy scheduler configuration and create new scheduler
    if hasattr(original_trainer, 'scheduler_y') and original_trainer.scheduler_y is not None:
        # Get the current state of the scheduler
        from functools import partial
        from torch.optim.lr_scheduler import LambdaLR
        
        if hasattr(original_trainer, 'lr') and original_trainer.lr is not None:
            cloned_trainer.scheduler_y = LambdaLR(
                cloned_trainer.optimizer_y, 
                partial(lr_lambda, warmup=1000, cycle_factor=0.8, cycle_length=1000)
            )

    
    # Set v to eval mode and freeze parameters
    cloned_trainer.v.eval()
    for param in cloned_trainer.v.parameters():
        param.requires_grad = False
    
    # Create a new file suffix to avoid overwriting original logs
    cloned_trainer.file_suffix = f"{original_trainer.file_suffix}_fixed_v_jl"
    
    # Start training loop with fixed v
    print(f"\nStarting additional training with fixed v for {additional_epochs} epochs...")
    print("Optimizing only J_l (energy functional) with fixed control")
    
    # Setup progress bar - now starting from 1
    pbar = tqdm(range(1, additional_epochs + 2), ascii=True, dynamic_ncols=True)
    
    # Training loop
    for epoch in pbar:
        # Set y to training mode
        cloned_trainer.y.train()
        
        # Train for one epoch
        total_loss = 0.0
        batch_count = 0
        
        for batch in cloned_trainer.train_loader:
            x = batch[0].detach().requires_grad_(True).to(cloned_trainer.device)
            
            # Only optimize for J_l loss
            loss_y = cloned_trainer.getloss_l(cloned_trainer.source, cloned_trainer.y, cloned_trainer.v, x)
            
            # Update y network
            cloned_trainer._update_model(cloned_trainer.y, cloned_trainer.optimizer_y, loss_y)
            
            total_loss += loss_y.item()
            batch_count += 1
        
        # Update learning rate
        if hasattr(cloned_trainer, 'scheduler_y') and cloned_trainer.scheduler_y is not None:
            cloned_trainer.scheduler_y.step()
        
        # Calculate average loss
        avg_loss = total_loss / batch_count if batch_count > 0 else 0
        
        # Record metrics periodically (without saving)
        if (epoch - 1) % 10 == 0:
            # Evaluate on grid for consistent metrics
            grid_size = 50
            x1 = torch.linspace(0, 1, grid_size, device=cloned_trainer.device)
            x2 = torch.linspace(0, 1, grid_size, device=cloned_trainer.device)
            X1, X2 = torch.meshgrid(x1, x2, indexing='ij')
            grid_points = torch.stack([X1.flatten(), X2.flatten()], dim=1).requires_grad_(True)
            
            
            # Clear cache periodically
            if epoch % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Update progress bar
        pbar.set_postfix(
            J_l_loss=avg_loss
        )
        
        # Validate periodically (just calculate error, no plotting)
        if (epoch-1) % 500 == 0:
            cloned_trainer.y.eval()
            rel_l2_y, rel_l2_u = rel_l2_error_yu(cloned_trainer.y, cloned_trainer.v, cloned_trainer.source, device=cloned_trainer.device)
            
            # Define CSV file path if not already defined
            if not hasattr(cloned_trainer, 'validation_csv_file'):
                validation_dir = os.path.join(ROOT, original_trainer.file_suffix)
                os.makedirs(validation_dir, exist_ok=True)
                cloned_trainer.validation_csv_file = os.path.join(validation_dir, 'forward_l2errors.csv')
                
                # Initialize the CSV file with header
                with open(cloned_trainer.validation_csv_file, 'w') as f:
                    f.write("epoch,rel_l2_error_y,rel_l2_error_u\n")
            
            # Append results to CSV
            with open(cloned_trainer.validation_csv_file, 'a') as f:
                f.write(f"{epoch},{rel_l2_y.item()},{rel_l2_u.item()}\n")
            
            
    
    # Update the total epochs
    cloned_trainer.num_epochs = additional_epochs
    

    
    return cloned_trainer

In [ ]:
cloned_trainer = clone_trainer_and_train_fixed_v(trainer, additional_epochs=10000)

In [ ]:
import os

# Save the final models in the models directory alongside loss.csv and norm.csv.
model_dir = os.path.join(ROOT, trainer.file_suffix, "models")
os.makedirs(model_dir, exist_ok=True)
torch.save(cloned_trainer.y.state_dict(), os.path.join(model_dir, "final_y_fixu.pth"))
torch.save(cloned_trainer.v.state_dict(), os.path.join(model_dir, "final_u_fixu.pth"))
torch.save(cloned_trainer.theta.state_dict(), os.path.join(model_dir, "final_theta_fixu.pth"))

# Final plots

In [ ]:
def plot_solutions(net_y, net_v, source, device='cuda', save_path=None):

    n_points = 200
    x = np.linspace(0, 1, n_points)
    y = np.linspace(0, 1, n_points)
    X, Y = np.meshgrid(x, y)
    

    xy = torch.tensor(np.stack([X.flatten(), Y.flatten()], axis=1), 
                     dtype=torch.float32, device=device)
    

    with torch.no_grad():
        y_pred = net_y(xy).cpu().numpy().reshape(n_points, n_points)
        v_pred = net_v(xy).cpu().numpy().reshape(n_points, n_points)
        y_exact = source.compute_y_star(xy).cpu().numpy().reshape(n_points, n_points)
        v_exact = source.compute_u_star(xy).cpu().numpy().reshape(n_points, n_points)
    

    error_y = np.abs(y_pred - y_exact)
    error_v = np.abs(v_pred - v_exact)
    save_dir = None
    if save_path:
        # treat save_path as directory if it is a dir, else use its dirname or itself
        if os.path.isdir(save_path):
            save_dir = save_path
        else:
            maybe_dir = os.path.dirname(save_path)
            save_dir = maybe_dir if maybe_dir else save_path
        os.makedirs(save_dir, exist_ok=True)
    def save_fig(name):
        if save_dir:
            plt.savefig(os.path.join(save_dir, f"{name}.pdf"), bbox_inches="tight")


    print("Numerical Solution - State y")
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X, Y, y_pred, cmap='coolwarm')
    ax.tick_params(axis='both', which='major', labelsize=14)
    cbar = plt.colorbar(surf, pad=0.1, fraction=0.046, shrink=1.0)
    cbar.ax.tick_params(labelsize=14)
    save_fig('y_numerical')
    plt.show()



    print("Numerical Solution - Control v")
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X, Y, v_pred, cmap='coolwarm')
    ax.tick_params(axis='both', which='major', labelsize=14) 
    

    cbar = plt.colorbar(surf, pad=0.1, fraction=0.046, shrink=1.0)
    cbar.ax.tick_params(labelsize=14)
    save_fig('u_numerical')
    plt.show()


# Create the directory for fixed solution comparisons
#solution_dir = os.path.join(ROOT, trainer.file_suffix, 'solution_comparison_fixed')
#os.makedirs(solution_dir, exist_ok=True)

# Plot the solutions with directory path for saving all plots
#plot_solutions(trainer.y, trainer.v, trainer.source, device=device,
#               save_path=os.path.join(solution_dir, 'solution'))

In [ ]:
# Relative L2 errors at different grid spacings h
# h = 1/(resolution-1), so resolution = 1/h + 1


denominators = [8, 16, 32, 64, 128,256,512,1024]

print("Grid spacing (h) | Resolution | rel_l2_error_y | rel_l2_error_u")
print("-" * 65)

for denom in denominators:
    h = 1.0 / denom

    resolution = int(1/h + 1)
    
    try:

        rel_l2_y, rel_l2_u = rel_l2_error_yv(
            cloned_trainer.y, 
            cloned_trainer.v, 
            cloned_trainer.source, 
            device=cloned_trainer.device,
            resolution=resolution
        )
        
        print(f"h = 1/{denom:3d}      | {resolution:10d} | {rel_l2_y.item():13.6e} | {rel_l2_u.item():13.6e}")
        
    except Exception as e:
        print(f"h = 1/{denom:3d}      | {resolution:10d} | Error: {str(e)}")

In [ ]:
# Load the fixu model and reproduce the final plots.

import os
# Load the fixed-control checkpoints saved above.
fixu_model_dir = os.path.join(ROOT, trainer.file_suffix, "models")


net_y_fixu = net_y_res(source=source, hidden=16, layers=4, activation='Swish').to(device)
net_v_fixu = net_v_res(source=source, hidden=16, layers=4, activation='Swish', hard_=hard, constraint=list(cloned_trainer.v.constraint)).to(device)
net_theta_fixu = net_y_res(source=source, hidden=16, layers=4, activation='Swish').to(device)


net_y_fixu.load_state_dict(torch.load(os.path.join(fixu_model_dir, "final_y_fixu.pth"), map_location=device))
net_v_fixu.load_state_dict(torch.load(os.path.join(fixu_model_dir, "final_u_fixu.pth"), map_location=device))
net_theta_fixu.load_state_dict(torch.load(os.path.join(fixu_model_dir, "final_theta_fixu.pth"), map_location=device))


net_y_fixu.eval()
net_v_fixu.eval()
net_theta_fixu.eval()


solution_dir_fixu = os.path.join(ROOT, 'solution_comparison_fixed_models')
os.makedirs(solution_dir_fixu, exist_ok=True)

print("Using fixu models for plotting...")
plot_solutions(net_y_fixu, net_v_fixu, source, device=device, save_path=os.path.join(solution_dir_fixu, 'solution_fixu'))